In [0]:
from pyspark.sql.functions import col, current_timestamp, from_unixtime
from pyspark.sql.types import DateType

# Read parquet file from cloud storage to Sparkk DataFrame
df = (spark.read
      .format('parquet')
      .load('/Volumes/workspace/dbacademy/samples/users-historical/')
      )

# Adding Metadata column, and ingestion_time column
df_with_metadata = (
    df.withColumn('first_touch_date', from_unixtime(col('user_first_touch_timestamp')/1000000).cast(DateType()))
    .withColumn('file_name',col('_metadata.file_name'))
    .withColumn('last_modified_time',col('_metadata.file_modification_time'))
    .withColumn('ingestion_time',current_timestamp())
)

# Writing Data from Dataframe into Table
(df_with_metadata.write
 .format('delta')
 .mode('append')
 .saveAsTable('dbacademy.users_historical_bronze_python')
)

# read and display the table
users_historical_bronze_python = spark.read.table('dbacademy.users_historical_bronze_python')
display(users_historical_bronze_python)